# Wordle: ES on top of prime-rl's Wordle-SFT checkpoint (week16)

**What changed vs week14:** load `PrimeIntellect/Qwen3-1.7B-Wordle-SFT` directly (skipping our own warm-start), drop the `char_head` + `vocab_trie` + per-letter feedback masks + autoregressive char loop, and rollout via `model.generate(...)` with `enable_thinking=False`. ES still runs on LoRA adapters only.

Three numbers come out of this notebook:
1. **`RL_ceiling`** — `PrimeIntellect/Qwen3-1.7B-Wordle-RL` evaluated on our env (their full pipeline)
2. **`SFT_cold`** — `PrimeIntellect/Qwen3-1.7B-Wordle-SFT` evaluated cold (the starting point for ES)
3. **`SFT + ES`** — our final checkpoint after the ES run

**Requires:** `torch`, `transformers>=4.51`, `peft>=0.10`, `jinja2>=3.1`, `numpy`, `matplotlib`. First run downloads the SFT and RL HF weights (≈7 GB combined).

## 0. GPU environment setup

Run once on a fresh GPU host. Skip if the kernel is already configured.

In [1]:
import os, sys, subprocess

def _pip_install(*pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *pkgs]
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd)

try:
    import peft  # noqa: F401
except ImportError:
    _pip_install('peft')

try:
    import jinja2
    _v = tuple(int(x) for x in jinja2.__version__.split('.')[:2])
    if _v < (3, 1):
        _pip_install("jinja2>=3.1.0")
except ImportError:
    _pip_install("jinja2>=3.1.0")

import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no CUDA device visible — generation-based ES will be unusably slow on CPU.')

torch: 2.11.0+cu130 | cuda: True
cuda_device: NVIDIA A100 80GB PCIe


## 1. Imports and device

Brings `src/` onto `sys.path`, imports `WordleQwenPolicy` (the new generation-based policy) and the unchanged `train_es_wordle` from `es_wordle`. No `train_curriculum`, no `wordle_gpt2_warmstart`, no `WordleGPT2Policy`.

In [2]:
import os
import sys
import gc
import importlib
from pathlib import Path

import numpy as np
import torch

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

_here = Path.cwd().resolve()
ROOT = _here.parent if _here.name == "notebooks" else _here
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wordle_env import load_wordle_environment
import wordle_qwen_policy
importlib.reload(wordle_qwen_policy)
from wordle_qwen_policy import WordleQwenPolicy

import es_wordle
importlib.reload(es_wordle)
from es_wordle import train_es_wordle, es_gradient_estimate_wordle, _rollout_batched


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps_backend = getattr(torch.backends, "mps", None)
    if mps_backend is not None and mps_backend.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def cuda_dtype_kwargs() -> dict:
    if not torch.cuda.is_available():
        return {}
    if torch.cuda.is_bf16_supported():
        return {"torch_dtype": torch.bfloat16}
    return {"torch_dtype": torch.float16}


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


DEVICE = choose_device()
MODEL_LOAD_KWARGS = cuda_dtype_kwargs()

print("ROOT:", ROOT)
print("es_wordle:", es_wordle.__file__)
print("wordle_qwen_policy:", wordle_qwen_policy.__file__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("device:", DEVICE, "| model_load_kwargs:", MODEL_LOAD_KWARGS or "{torch_dtype: float32}")
if torch.cuda.is_available():
    print("cuda_device:", torch.cuda.get_device_name(0))

ROOT: /home/ubuntu/STAT-4830-project-base
es_wordle: /home/ubuntu/STAT-4830-project-base/src/es_wordle.py
wordle_qwen_policy: /home/ubuntu/STAT-4830-project-base/src/wordle_qwen_policy.py
torch: 2.11.0+cu130 | cuda: True
device: cuda | model_load_kwargs: {'torch_dtype': torch.bfloat16}
cuda_device: NVIDIA A100 80GB PCIe


## 2. Hyperparameters

**Thinking is on** because the SFT/RL Wordle checkpoints were trained to emit `<think>...</think>` blocks regardless of the chat-template flag. First run with `enable_thinking=False`, `max_new_tokens=64` produced 100% parse failure (the model used the whole budget on the thinking opener and never reached `<guess>`).

With thinking enabled at 512 tokens, wall-clock per ES iter is roughly 8× higher than the original estimate, so the run scale is **reduced to `N_POP=8`, `N_ITER=15`** for the first attempt. Run §2.5 first; if the smoke test passes, do a single-iter dry run from §3 to measure wall-clock before letting the full schedule complete.

In [ ]:
import random

SEED = 42

# === Models ============================================================
MODEL_SFT = "PrimeIntellect/Qwen3-1.7B-Wordle-SFT"
MODEL_RL_REF = "PrimeIntellect/Qwen3-1.7B-Wordle-RL"

# === Generation ========================================================
# enable_thinking=True is required: the SFT checkpoint always emits <think> blocks
# (training data forced the format), and 64-token budgets truncate before <guess>.
MAX_PROMPT_LENGTH = 1024
ENABLE_THINKING = True
MAX_NEW_TOKENS = 512
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.9

# === LoRA ==============================================================
LORA_R = 4
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
CAST_LORA_TO_FP32 = True  # cleaner ES gradient math; bf16 base stays bf16

# === ES hyperparameters ================================================
# Scaled down from N_POP=16/N_ITER=30 because thinking-on generation is ~8x
# slower per turn than the original estimate. Do a single-iter dry run from
# §3 to measure wall-clock before committing to the full schedule.
N_POP = 8
N_ITER = 15
N_EVAL_EPISODES = 8    # episodes per pop-member fitness eval (was 16)
EVAL_N_EPISODES = 16   # episodes per periodic full eval (was 32)
EVAL_EVERY = 1
PROBE_N_EPISODES = 8   # was 16
MAX_TURNS = 6
SIGMA = 0.02
ALPHA = None  # auto-calibrated below from one-shot gradient probe
NORMALIZE_GRADIENT = False
RANK_FITNESS = True
BASELINE_SUBTRACT = True  # PGPE-lite; takes precedence over rank_fitness
ANTITHETIC = True
COMMON_RANDOM_NUMBERS = True
EMA_BETA = 0.0
EVAL_DETERMINISTIC = True
FITNESS_OBJECTIVE = "win_plus_return"
WIN_FITNESS_SCALE = 8.0
PER_ITER_SECRET_SUBSET_SIZE = 8  # mini-batch ES under CRN; matched to N_EVAL_EPISODES
TRACK_BEST_ITER = True
RESTORE_BEST_ON_FINISH = False

# === Reference eval ====================================================
REF_EVAL_EPISODES = 100
SAMPLE_ROLLOUT_COUNT = 4

# === Env ===============================================================
MOCK_ENV = False
NUM_TRAIN_EXAMPLES = 2000
NUM_EVAL_EXAMPLES = 100

# === RNG ===============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(
    f"Models:\n  SFT (start)  : {MODEL_SFT}\n  RL  (ceiling): {MODEL_RL_REF}\n"
    f"Gen: enable_thinking={ENABLE_THINKING}  max_new_tokens={MAX_NEW_TOKENS}  "
    f"temperature={GEN_TEMPERATURE}  top_p={GEN_TOP_P}\n"
    f"LoRA: r={LORA_R}  alpha={LORA_ALPHA}  targets={LORA_TARGET_MODULES}  "
    f"cast_to_fp32={CAST_LORA_TO_FP32}\n"
    f"ES: N_POP={N_POP}  N_ITER={N_ITER}  n_eval_eps={N_EVAL_EPISODES}  "
    f"eval_eps={EVAL_N_EPISODES}  eval_every={EVAL_EVERY}\n"
    f"  fitness={FITNESS_OBJECTIVE} (scale={WIN_FITNESS_SCALE})  "
    f"baseline_subtract={BASELINE_SUBTRACT}  antithetic={ANTITHETIC}  CRN={COMMON_RANDOM_NUMBERS}\n"
    f"  per_iter_secret_subset_size={PER_ITER_SECRET_SUBSET_SIZE}  ALPHA=auto"
)

## 2.5 Reference eval and smoke test

Three jobs in this section:
1. **`RL_ceiling`** — evaluate `PrimeIntellect/Qwen3-1.7B-Wordle-RL` on 100 episodes of our env. This is what their full pipeline scores in *our* setup; it tells us if our env is faithful to theirs.
2. **`SFT_cold`** — evaluate `PrimeIntellect/Qwen3-1.7B-Wordle-SFT` cold on 100 episodes. This is the starting point ES has to improve on.
3. **Smoke test** — print 4 sample rollouts verbatim from the SFT model. Confirms `enable_thinking=False` produces parseable `<guess>WORD</guess>` output. If parse-success is bad, switch to `enable_thinking=True, max_new_tokens=256` before §3.

**Sanity gates** (per the plan): RL ceiling ≥ 30%, SFT cold ≥ 25%, parse-success ≥ 90% on the 4 samples. If any fail, stop and diagnose.

In [4]:
def quick_eval_model(
    model_name: str,
    env,
    n_episodes: int,
    enable_thinking: bool = False,
    max_new_tokens: int = 64,
    deterministic: bool = True,
    keep_loaded: bool = False,
):
    """Load `model_name`, run `n_episodes` deterministic rollouts on `env`,
    return (win_rate, mean_reward, mean_turns, parse_failure_rate, oov_rate, policy_or_None).

    Set `keep_loaded=True` to return the policy without freeing it (caller is
    responsible for `del policy; free_gpu()` when done).
    """
    print(f"\n=== Eval: {model_name} | episodes={n_episodes} | thinking={enable_thinking} ===")
    policy = WordleQwenPolicy(
        model_name=model_name,
        max_prompt_length=MAX_PROMPT_LENGTH,
        enable_thinking=enable_thinking,
        max_new_tokens=max_new_tokens,
        gen_temperature=GEN_TEMPERATURE,
        gen_top_p=GEN_TOP_P,
        use_lora=False,  # eval the raw checkpoint
        cast_lora_to_fp32=False,
        model_kwargs=MODEL_LOAD_KWARGS,
    ).to(DEVICE)
    print(f"  total params: {policy.count_parameters():,}")
    print(f"  trainable   : {policy.count_trainable_parameters():,} (frozen for ref eval)")

    policy.reset_trie_stats()
    rewards, successes, turns = _rollout_batched(
        policy, env, n_episodes=n_episodes, max_turns=MAX_TURNS,
        deterministic=deterministic,
    )
    win_rate = sum(1 for s in successes if s >= 0.5) / max(1, len(successes))
    mean_r = float(np.mean(rewards)) if rewards else float("nan")
    mean_t = float(np.mean(turns)) if turns else float("nan")
    stats = policy.trie_stats()
    parse_fail = float(stats.get("trie_fallback_rate", float("nan")))
    oov_rate = (
        stats["oov_words"] / stats["trie_steps"] if stats["trie_steps"] > 0 else float("nan")
    )
    print(
        f"  win_rate={win_rate:.1%}  mean_reward={mean_r:+.3f}  mean_turns={mean_t:.2f}  "
        f"parse_fail_rate={parse_fail:.1%}  oov_rate={oov_rate:.1%}"
    )

    if not keep_loaded:
        del policy
        free_gpu()
        return win_rate, mean_r, mean_t, parse_fail, oov_rate, None
    return win_rate, mean_r, mean_t, parse_fail, oov_rate, policy


def print_sample_rollouts(policy, env, n: int = 4):
    """Run `n` deterministic episodes and print every turn's prompt + raw model output
    + parsed guess + env feedback. Used to inspect generation behavior."""
    from wordle_qwen_policy import _build_wordle_prompt
    print(f"\n--- {n} sample rollouts (deterministic) ---")
    for ep in range(n):
        state = env.reset()
        target = state.target_word
        print(f"\n[ep {ep+1}] target={target}")
        done = False
        turn = 0
        info = {}
        while not done and turn < MAX_TURNS:
            # Build the same prompt the policy will see and generate one rollout's text.
            enc = policy._build_chat_inputs([state])
            input_len = int(enc["input_ids"].shape[1])
            with torch.no_grad():
                out = policy.lm.generate(
                    **enc,
                    max_new_tokens=policy.max_new_tokens,
                    do_sample=False,
                    pad_token_id=policy.tokenizer.pad_token_id,
                    eos_token_id=policy.tokenizer.eos_token_id,
                )
            gen_text = policy.tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)[0]
            word, parsed_clean = policy._parse_word_from_text(gen_text)
            xml = f"<think>(decoded)</think>\n<guess>{word}</guess>"
            env.current_state = state
            state, _r, done, info = env.step(xml)
            turn += 1
            short = gen_text[:160].replace("\n", " \u23ce ")
            print(f"  t{turn} parsed={word} clean={parsed_clean} | gen[:160]={short!r}")
            print(f"     feedback: {info.get('feedback')}")
        print(f"  result: correct_answer={info.get('correct_answer'):.0f}  turns={turn}")


print("eval helpers ready")

eval helpers ready


In [5]:
# Build the env once. Same 2,315-answer pool used throughout.
ref_env = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
)

# 1) RL ceiling (their full pipeline)
RL_ceiling, RL_mean_reward, RL_mean_turns, RL_parse_fail, RL_oov, _ = quick_eval_model(
    MODEL_RL_REF, ref_env, REF_EVAL_EPISODES,
    enable_thinking=ENABLE_THINKING, max_new_tokens=MAX_NEW_TOKENS,
    deterministic=True, keep_loaded=False,
)

# 2) SFT cold baseline (the starting point for ES)
SFT_cold, SFT_mean_reward, SFT_mean_turns, SFT_parse_fail, SFT_oov, sft_policy_for_smoke = (
    quick_eval_model(
        MODEL_SFT, ref_env, REF_EVAL_EPISODES,
        enable_thinking=ENABLE_THINKING, max_new_tokens=MAX_NEW_TOKENS,
        deterministic=True, keep_loaded=True,  # keep for sample printing below
    )
)

# 3) Smoke test: print 4 verbatim rollouts from the SFT model.
print_sample_rollouts(sft_policy_for_smoke, ref_env, n=SAMPLE_ROLLOUT_COUNT)

# Free the SFT smoke-test policy. §3 will reinstantiate with LoRA enabled.
del sft_policy_for_smoke
free_gpu()

print(
    f"\n=== Reference eval summary ===\n"
    f"  RL_ceiling : {RL_ceiling:>6.1%}  (mean_reward={RL_mean_reward:+.3f}, mean_turns={RL_mean_turns:.2f})\n"
    f"  SFT_cold   : {SFT_cold:>6.1%}  (mean_reward={SFT_mean_reward:+.3f}, mean_turns={SFT_mean_turns:.2f})\n"
    f"  parse fail : SFT={SFT_parse_fail:.1%}  RL={RL_parse_fail:.1%}\n"
    f"  oov words  : SFT={SFT_oov:.1%}  RL={RL_oov:.1%}"
)

# Sanity gates from the plan.
_gate_ok = True
if RL_ceiling < 0.30:
    print("  [GATE] RL_ceiling < 30% — env may be misaligned with prime-rl's setup. Investigate.")
    _gate_ok = False
if SFT_cold < 0.25:
    print("  [GATE] SFT_cold < 25% — generation/parsing likely broken. Do not start ES.")
    _gate_ok = False
if SFT_parse_fail > 0.10 or (SFT_parse_fail != SFT_parse_fail):  # >10% or NaN
    print("  [GATE] parse-failure rate > 10% — try ENABLE_THINKING=True, MAX_NEW_TOKENS=256 and re-run §2.5.")
    _gate_ok = False
if _gate_ok:
    print("  [GATE] all checks passed — ready for §3.")
else:
    print("  [GATE] one or more sanity checks failed — see messages above before continuing.")

[INFO] verifiers package not installed; using bundled Wordle answer list.
[OK] Local Wordle environment ready (2315 target words).

=== Eval: PrimeIntellect/Qwen3-1.7B-Wordle-RL | episodes=100 | thinking=False ===


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

  total params: 1,720,574,976
  trainable   : 0 (frozen for ref eval)
  win_rate=1.0%  mean_reward=+0.281  mean_turns=5.98  parse_fail_rate=100.0%  oov_rate=0.0%

--- 4 sample rollouts (deterministic) ---

[ep 1] target=FLUNK
  t1 parsed=THINK clean=False | gen[:160]="<think> ⏎ I'm playing Wordle and need to choose a strong opening word. A good starting word should have common vowels and consonants to maximize information gain."
     feedback: T:GRAY H:GRAY I:GRAY N:GREEN K:GREEN
  t2 parsed=THINK clean=False | gen[:160]='<think> ⏎ The previous guess was "THINK" with feedback: ⏎ - T:GRAY ⏎ - H:GRAY ⏎ - I:GRAY ⏎ - N:GREEN ⏎ - K:GREEN ⏎  ⏎ From this, we know: ⏎ 1. The word has the pattern ???NK (sin'
     feedback: T:GRAY H:GRAY I:GRAY N:GREEN K:GREEN
  t3 parsed=THINK clean=False | gen[:160]='<think> ⏎ The previous guesses for Wordle are: ⏎ 1. THINK: T:GRAY H:GRAY I:GRAY N:GREEN K:GREEN ⏎ 2. THINK: T:GRAY H:GRAY I:GRAY N:GREEN K:GREEN ⏎  ⏎ From this, we can de'
     feedback: T:GRAY H:G

## 3. ES training

Reinstantiate the SFT model with LoRA enabled, calibrate `ALPHA` from a one-shot ES gradient probe (target initial step ≈ 0.13), then run `train_es_wordle` for `N_ITER=30` iterations. The ES code is unchanged from week14; the only differences are the policy class and the absence of curriculum / warm-start.

**Diagnostics column meanings** (verbose train log):
- **TRAIN**: `Fit` mean fitness across population, `ES_win` mean win rate, `popσ` fitness spread
- **EVAL**: held-out `Succ`, mean reward, mean turns (greedy)
- **OPT**: `Δθ` total drift from init, `|g|` raw gradient norm, `|Δ|` applied step norm, `cos` cos similarity to previous gradient
- **SIG**: `ess` unique-fitness count out of N (collapses to 1 when most members tie at 0 wins), `wins` count of winning population members, `dprobe` post−pre delta on a fixed probe slate, `fb%` parse-failure rate (repurposed from the trie-fallback hook)

In [ ]:
# Reseed for reproducibility before building the LoRA policy.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

policy = WordleQwenPolicy(
    model_name=MODEL_SFT,
    max_prompt_length=MAX_PROMPT_LENGTH,
    enable_thinking=ENABLE_THINKING,
    max_new_tokens=MAX_NEW_TOKENS,
    gen_temperature=GEN_TEMPERATURE,
    gen_top_p=GEN_TOP_P,
    use_lora=True,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_target_modules=LORA_TARGET_MODULES,
    cast_lora_to_fp32=CAST_LORA_TO_FP32,
    model_kwargs=MODEL_LOAD_KWARGS,
).to(DEVICE)

print(
    f"Trainable: {policy.count_trainable_parameters():,}  |  "
    f"Total: {policy.count_parameters():,}  |  "
    f"action_granularity={policy.action_granularity}  |  "
    f"action_dim={policy.action_dim}"
)
_first_trainable = next((p for p in policy.parameters() if p.requires_grad), None)
if _first_trainable is not None:
    print(f"  first trainable param: dtype={_first_trainable.dtype}  shape={tuple(_first_trainable.shape)}")

# Sanity: trainable count should be ~0.5–1.5M, not 1.7B.
_trainable = policy.count_trainable_parameters()
if _trainable > 50_000_000:
    print(f"  [WARN] trainable params = {_trainable:,} — LoRA freezing may have failed; expected <2M.")

# Build the two envs: env_train for ES rollouts, env_eval for periodic eval.
env_train = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
)
env_eval = load_wordle_environment(
    num_train_examples=NUM_TRAIN_EXAMPLES,
    num_eval_examples=NUM_EVAL_EXAMPLES,
    use_prime_intellect=not MOCK_ENV,
)
print("envs built; ES is on env_train, periodic eval is on env_eval (independent secret draws).")

In [ ]:
# === ALPHA calibration probe ===========================================
# One-shot ES gradient estimate at init. Pick ALPHA so the initial step is
# ~0.13 in parameter-space norm. RNG state is snapshotted/restored so the
# probe does not perturb the training stream.
_target_step = 0.13
_min_step = 0.05

_py_state = random.getstate()
_np_state = np.random.get_state()
_torch_state = torch.get_rng_state()
_cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

try:
    _grad, _avg_fit, _fits, _avg_win, _pop_std, _es_wrs = es_gradient_estimate_wordle(
        policy,
        env_train,
        N=N_POP,
        sigma=SIGMA,
        n_eval_episodes=N_EVAL_EPISODES,
        max_turns=MAX_TURNS,
        rank_fitness=RANK_FITNESS,
        fitness_objective=FITNESS_OBJECTIVE,
        win_fitness_scale=WIN_FITNESS_SCALE,
        antithetic=ANTITHETIC,
        common_random_numbers=COMMON_RANDOM_NUMBERS,
        baseline_subtract=BASELINE_SUBTRACT,
        per_iter_secret_subset_size=PER_ITER_SECRET_SUBSET_SIZE,
    )
    _g_norm = float(_grad.norm().item())
    _ess_rank = len({round(float(f), 6) for f in _fits})
    _win_count = sum(1 for wr in _es_wrs if wr > 0.0)
    if _g_norm <= 1e-8:
        raise RuntimeError(
            "ALPHA auto-calibration failed: probe ‖ĝ‖ ≈ 0. No ES signal at all — likely a "
            "generation/parsing bug or the SFT model is too consistent at this scale."
        )
    _suggested_alpha = float(_target_step / _g_norm)
    if ALPHA is None:
        ALPHA = _suggested_alpha
        _action = f"  AUTO ALPHA = {ALPHA:.2e}  (initial step ≈ {_target_step})"
    else:
        _action = f"  manual ALPHA = {ALPHA:.2e}  (calibrator suggested {_suggested_alpha:.2e})"
    _implied_step = ALPHA * _g_norm
    print(
        f"\n[ALPHA calibration @ init, trainable={policy.count_trainable_parameters():,}]\n"
        f"  raw ‖ĝ‖           = {_g_norm:.4g}\n"
        f"  ALPHA * ‖ĝ‖       = {_implied_step:.4g}   (target ≈ {_target_step})\n"
        f"  ES probe avg_fit  = {_avg_fit:+.3f}   ES_win = {_avg_win:.1%}   popσ = {_pop_std:.4f}\n"
        f"  ES signal probes  = ess_rank {_ess_rank}/{N_POP}, wins {_win_count}/{N_POP}\n"
        f"{_action}"
    )
    if _implied_step < _min_step:
        print(
            f"  ⚠ implied step {_implied_step:.4g} < {_min_step}. Suggested ALPHA ≈ "
            f"{_suggested_alpha:.2e}; setting ALPHA=None re-runs the probe to apply it."
        )
finally:
    random.setstate(_py_state)
    np.random.set_state(_np_state)
    torch.set_rng_state(_torch_state)
    if _cuda_state is not None:
        torch.cuda.set_rng_state_all(_cuda_state)
    del _grad, _avg_fit, _fits, _avg_win, _pop_std, _es_wrs
    free_gpu()

In [ ]:
history = train_es_wordle(
    policy,
    env_train,
    N=N_POP,
    sigma=SIGMA,
    alpha=ALPHA,
    n_iterations=N_ITER,
    n_eval_episodes=N_EVAL_EPISODES,
    max_turns=MAX_TURNS,
    eval_every=EVAL_EVERY,
    verbose=True,
    normalize_gradient=NORMALIZE_GRADIENT,
    eval_n_episodes=EVAL_N_EPISODES,
    rank_fitness=RANK_FITNESS,
    eval_deterministic=EVAL_DETERMINISTIC,
    fitness_objective=FITNESS_OBJECTIVE,
    win_fitness_scale=WIN_FITNESS_SCALE,
    antithetic=ANTITHETIC,
    common_random_numbers=COMMON_RANDOM_NUMBERS,
    ema_beta=EMA_BETA,
    env_eval=env_eval,
    probe_n_episodes=PROBE_N_EPISODES,
    baseline_subtract=BASELINE_SUBTRACT,
    per_iter_secret_subset_size=PER_ITER_SECRET_SUBSET_SIZE,
    track_best_iter=TRACK_BEST_ITER,
    restore_best_on_finish=RESTORE_BEST_ON_FINISH,
)

final_eval_success = float(history["eval_success"][-1]) if history.get("eval_success") else float("nan")
best_eval_success = (
    float(history["best_eval_success"][0])
    if history.get("best_eval_success") and history["best_eval_success"][0] == history["best_eval_success"][0]
    else float("nan")
)
best_iter = int(history["best_iter"][0]) if history.get("best_iter") else -1
print(
    f"\n=== ES finished ===\n"
    f"  final eval_success  : {final_eval_success:.1%}\n"
    f"  best  eval_success  : {best_eval_success:.1%} (iter {best_iter})\n"
    f"  ES contribution     : {(final_eval_success - SFT_cold):+.1%}  (final − SFT_cold)\n"
    f"  Gap to RL ceiling   : {(RL_ceiling - final_eval_success):+.1%}  (RL_ceiling − final)"
)

## 4. Plot training curves

Same panel layout as week14. The trie-stats panels (parse-failure rate, parse-attempts, OOV) now reflect the *generation*-based diagnostics the new policy reports instead of the legacy char-trie counters.

In [ ]:
import matplotlib.pyplot as plt

it = np.array(history.get("iteration", []), dtype=float)
ti = np.array(history.get("train_iter", []), dtype=float)

fig, axes = plt.subplots(4, 3, figsize=(16, 12), sharex=False)

# Core eval metrics
axes[0, 0].plot(it, history.get("eval_success", []), "g-o", ms=3)
axes[0, 0].axhline(SFT_cold, color="gray", ls=":", lw=0.9, label=f"SFT_cold={SFT_cold:.1%}")
axes[0, 0].axhline(RL_ceiling, color="red", ls=":", lw=0.9, label=f"RL_ceiling={RL_ceiling:.1%}")
axes[0, 0].set_title("Eval success rate")
axes[0, 0].set_xlabel("iteration")
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].legend(loc="lower right", fontsize=7)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(it, history.get("eval_reward", []), "b-o", ms=3)
axes[0, 1].set_title("Eval mean reward")
axes[0, 1].set_xlabel("iteration")
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(it, history.get("eval_turns", []), "m-o", ms=3)
axes[0, 2].set_title("Eval mean turns")
axes[0, 2].set_xlabel("iteration")
axes[0, 2].grid(True, alpha=0.3)

# Optimization mechanics
axes[1, 0].plot(ti, history.get("train_fitness", []), color="c", lw=1)
axes[1, 0].set_title("ES mean fitness (per iter)")
axes[1, 0].set_xlabel("iteration")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(ti, history.get("train_grad_norm", []), color="r", lw=1)
axes[1, 1].set_title("Applied gradient norm")
axes[1, 1].set_xlabel("iteration")
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(ti, history.get("param_drift", []), color="k", lw=1)
axes[1, 2].set_title("Parameter drift ‖θ − θ₀‖")
axes[1, 2].set_xlabel("iteration")
axes[1, 2].grid(True, alpha=0.3)

# Signal quality
axes[2, 0].plot(ti, history.get("pop_fitness_std", []), color="orange", lw=1)
axes[2, 0].set_title("Population fitness std")
axes[2, 0].set_xlabel("iteration")
axes[2, 0].grid(True, alpha=0.3)

gc_arr = np.array(history.get("train_grad_cos", []), dtype=float)
axes[2, 1].plot(ti, gc_arr, color="purple", lw=1)
axes[2, 1].axhline(0.0, color="gray", lw=0.7, ls=":")
axes[2, 1].set_title("Gradient cosine cos(g_t, g_{t-1})")
axes[2, 1].set_xlabel("iteration")
axes[2, 1].set_ylim(-1.05, 1.05)
axes[2, 1].grid(True, alpha=0.3)

ess_arr = np.array(history.get("train_ess_rank", []), dtype=float)
axes[2, 2].plot(ti, ess_arr, color="teal", lw=1)
axes[2, 2].axhline(N_POP, color="gray", lw=0.7, ls=":", label=f"N_POP={N_POP}")
axes[2, 2].set_title("ESS rank (unique fitness count)")
axes[2, 2].set_xlabel("iteration")
axes[2, 2].set_ylim(0, max(2, N_POP + 1))
axes[2, 2].legend(loc="lower right", fontsize=7)
axes[2, 2].grid(True, alpha=0.3)

# Probe + parsing diagnostics
probe_delta = np.array(history.get("train_probe_delta", []), dtype=float)
finite_probe = np.isfinite(probe_delta)
axes[3, 0].axhline(0.0, color="gray", lw=0.7, ls=":")
axes[3, 0].plot(ti[finite_probe], probe_delta[finite_probe], "o-", ms=3, color="firebrick", lw=1)
axes[3, 0].set_title("Δ probe success (post − pre)")
axes[3, 0].set_xlabel("iteration")
axes[3, 0].grid(True, alpha=0.3)

fb_rate = np.array(history.get("train_trie_fallback_rate", []), dtype=float)
axes[3, 1].plot(ti, fb_rate, color="brown", lw=1)
axes[3, 1].set_title("Parse-failure rate (gen-mode)")
axes[3, 1].set_xlabel("iteration")
axes[3, 1].set_ylim(0, 1.05)
axes[3, 1].grid(True, alpha=0.3)

parse_attempts = np.array(history.get("train_trie_steps", []), dtype=float)
oov = np.array(history.get("train_trie_oov_words", []), dtype=float)
axes[3, 2].plot(ti, parse_attempts, color="slateblue", lw=1, label="parse attempts")
axes[3, 2].plot(ti, oov, color="darkred", lw=1, label="oov words")
axes[3, 2].set_title("Generation volume")
axes[3, 2].set_xlabel("iteration")
axes[3, 2].legend(loc="upper right", fontsize=7)
axes[3, 2].grid(True, alpha=0.3)

plt.suptitle(
    f"Wordle ES on SFT | {MODEL_SFT} | LoRA r={LORA_R} | thinking={ENABLE_THINKING}"
)
plt.tight_layout()
plt.show()

## 5. Final attribution and save

The three headline numbers and the ES contribution. Optionally save the LoRA adapter.

In [ ]:
es_contribution = final_eval_success - SFT_cold
gap_to_ceiling = RL_ceiling - final_eval_success

print("=" * 60)
print("FINAL ATTRIBUTION")
print("=" * 60)
print(f"  RL_ceiling (their full pipeline) : {RL_ceiling:>6.1%}")
print(f"  SFT_cold   (start of ES)         : {SFT_cold:>6.1%}")
print(f"  SFT + ES   (final)               : {final_eval_success:>6.1%}")
print(f"  SFT + ES   (best iter {best_iter:>3d})        : {best_eval_success:>6.1%}")
print("-" * 60)
print(f"  ES contribution                  : {es_contribution:+6.1%}  (final − SFT_cold)")
print(f"  Gap to ceiling                   : {gap_to_ceiling:+6.1%}  (RL_ceiling − final)")
print("=" * 60)

if es_contribution < 0:
    print(
        "\nNote: ES did not improve on the cold SFT checkpoint. Plausible causes:\n"
        "  - SFT is a tuned local minimum; perturbations regress capability faster than they help.\n"
        "  - Try smaller SIGMA / ALPHA, or set RESTORE_BEST_ON_FINISH=True and report best_iter."
    )

In [ ]:
save_path = ROOT / "models" / "wordle_qwen_es_lora.week16"
save_path.parent.mkdir(parents=True, exist_ok=True)
try:
    policy.lm.save_pretrained(str(save_path))
    print(f"Saved LoRA adapter to: {save_path}")
except Exception as exc:
    print(f"[skip] adapter save failed ({exc.__class__.__name__}: {exc}); not fatal.")

# Lightweight history dump (drop the large best_params tensor before pickling).
try:
    import pickle
    history_save = {k: v for k, v in history.items() if k != "best_params"}
    history_path = ROOT / "models" / "wordle_qwen_es_history.week16.pkl"
    with history_path.open("wb") as f:
        pickle.dump({
            "history": history_save,
            "RL_ceiling": RL_ceiling,
            "SFT_cold": SFT_cold,
            "final_eval_success": final_eval_success,
            "best_eval_success": best_eval_success,
            "best_iter": best_iter,
            "hparams": {
                "MODEL_SFT": MODEL_SFT, "MODEL_RL_REF": MODEL_RL_REF,
                "LORA_R": LORA_R, "N_POP": N_POP, "N_ITER": N_ITER,
                "SIGMA": SIGMA, "ALPHA": ALPHA,
                "ENABLE_THINKING": ENABLE_THINKING, "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
                "PER_ITER_SECRET_SUBSET_SIZE": PER_ITER_SECRET_SUBSET_SIZE,
            },
        }, f)
    print(f"Saved history to: {history_path}")
except Exception as exc:
    print(f"[skip] history pickle failed ({exc.__class__.__name__}: {exc}); not fatal.")